# RespiraTwin: Exploratory Data Analysis & Physiological Sanity Checks

> **CRITICAL DISCLAIMER**: Event labels and outcomes in RespiraTwin are generated by a simulator. RespiraTwin does **not** claim clinical accuracy. Results measure the learning of a simulated process, not real-world clinical prediction. It is intended strictly for research and proof-of-concept.

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Set plotting style
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10

# Locate datasets relative to notebook or workspace root
base_dir = os.path.dirname(os.path.abspath(''))
data_dir = os.path.join(base_dir, 'data')
if not os.path.exists(data_dir):
    data_dir = 'data'

patients_df = pd.read_csv(os.path.join(data_dir, 'patients.csv'))
pollution_df = pd.read_csv(os.path.join(data_dir, 'sample_pollution.csv'))
ts_df = pd.read_csv(os.path.join(data_dir, 'patient_timeseries.csv'))

print(f"Loaded {len(patients_df)} patient profiles.")
print(f"Loaded {len(pollution_df)} ambient pollution hourly records.")
print(f"Loaded {len(ts_df)} wearable time-series records for {ts_df['patient_id'].nunique()} patients.")

## 1. Patient EHR Cohort Summary
Let us examine disease severity breakdown, baseline vitals, and pollution sensitivity phenotypes.

In [ ]:
print("Asthma Severity Breakdown:")
print(patients_df['asthma_severity'].value_counts(normalize=True).round(3))

print("\nPollution Sensitivity Breakdown:")
print(patients_df['pollution_sensitivity'].value_counts(normalize=True).round(3))

print("\nPrior 12m Exacerbations by Severity:")
print(patients_df.groupby('asthma_severity')['prior_exacerbations_12m'].mean().round(2))

patients_df.head()

## 2. Ambient Air Quality Dynamics (PM2.5, PM10, AQI)
Notice that PM2.5 and PM10 concentrations (ug/m3) are tracked independently from the official calculated AQI index.

In [ ]:
pollution_df['timestamp'] = pd.to_datetime(pollution_df['timestamp'])
plot_poll = pollution_df.iloc[:24*14]  # First 14 days

fig, ax1 = plt.subplots(figsize=(14, 4))
ax1.plot(plot_poll['timestamp'], plot_poll['pm25'], label='PM2.5 (ug/m3)', color='#e74c3c', lw=1.5)
ax1.plot(plot_poll['timestamp'], plot_poll['pm10'], label='PM10 (ug/m3)', color='#e67e22', lw=1.2, alpha=0.7)
ax1.set_ylabel('Concentration (ug/m3)', color='#c0392b')
ax1.set_title('Ambient Air Quality Dynamics (First 14 Days - Delhi Benchmark)')

ax2 = ax1.twinx()
ax2.plot(plot_poll['timestamp'], plot_poll['aqi'], label='US EPA AQI Index', color='#8e44ad', linestyle='--', lw=1.2)
ax2.set_ylabel('AQI Index', color='#8e44ad')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
plt.grid(True, alpha=0.3)
plt.show()

## 3. Physiological Sanity Check: High vs. Low Pollution Sensitivity
We compare two distinct virtual patients during the same environmental exposure window:
- Patient A (High sensitivity, Severe asthma)
- Patient B (Low sensitivity, Mild asthma)

We observe:
1. Circadian rhythm in resting heart rate and respiratory rate.
2. Delayed respiratory rate and symptom elevation following prolonged high PM2.5 exposure in the sensitive patient.
3. Reactive rescue inhaler puff actuations accompanying symptom spikes.

In [ ]:
ts_df['timestamp'] = pd.to_datetime(ts_df['timestamp'])

# Find representative high and low sensitivity patients
high_sens_id = patients_df[patients_df['pollution_sensitivity'] == 'High']['patient_id'].iloc[0]
low_sens_id = patients_df[patients_df['pollution_sensitivity'] == 'Low']['patient_id'].iloc[0]

print(f"Comparing High Sensitivity Patient ({high_sens_id}) vs Low Sensitivity Patient ({low_sens_id})")

df_high = ts_df[ts_df['patient_id'] == high_sens_id].iloc[:24*14]
df_low = ts_df[ts_df['patient_id'] == low_sens_id].iloc[:24*14]

fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)

# 1. Respiratory Rate
axes[0].plot(df_high['timestamp'], df_high['respiratory_rate'], label=f'{high_sens_id} (High Sens) RR', color='#e74c3c')
axes[0].plot(df_low['timestamp'], df_low['respiratory_rate'], label=f'{low_sens_id} (Low Sens) RR', color='#2980b9', alpha=0.8)
axes[0].set_ylabel('Resp Rate (bpm)')
axes[0].legend(loc='upper right')
axes[0].set_title('Physiological Response Over 14 Days')
axes[0].grid(True, alpha=0.3)

# 2. SpO2 (%)
axes[1].plot(df_high['timestamp'], df_high['spo2'], label=f'{high_sens_id} SpO2 (%)', color='#c0392b')
axes[1].plot(df_low['timestamp'], df_low['spo2'], label=f'{low_sens_id} SpO2 (%)', color='#27ae60', alpha=0.8)
axes[1].set_ylabel('SpO2 (%)')
axes[1].legend(loc='lower right')
axes[1].grid(True, alpha=0.3)

# 3. Symptom Diary Score (0-10)
axes[2].step(df_high['timestamp'], df_high['symptom_score'], label=f'{high_sens_id} Symptoms (0-10)', color='#d35400', where='mid')
axes[2].step(df_low['timestamp'], df_low['symptom_score'], label=f'{low_sens_id} Symptoms (0-10)', color='#16a085', where='mid', alpha=0.8)
axes[2].set_ylabel('Symptom Score')
axes[2].legend(loc='upper right')
axes[2].grid(True, alpha=0.3)

# 4. Rescue Puffs Actuations
axes[3].bar(df_high['timestamp'], df_high['rescue_puffs'], width=0.03, label=f'{high_sens_id} Rescue Puffs', color='#8e44ad')
axes[3].bar(df_low['timestamp'], df_low['rescue_puffs'], width=0.03, label=f'{low_sens_id} Rescue Puffs', color='#3498db', alpha=0.5)
axes[3].set_ylabel('Puffs Count')
axes[3].set_xlabel('Timestamp')
axes[3].legend(loc='upper right')
axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Signal Coherence Verification
Confirm that across the cohort:
1. Rescue puff counts correlate with elevated symptom scores.
2. Sensor dropouts (NaNs) realistically model charging and off-wrist periods.

In [ ]:
print("Mean Rescue Puffs by Symptom Score Level:")
puffs_by_sym = ts_df.groupby('symptom_score')['rescue_puffs'].mean()
print(puffs_by_sym.round(2))

print("\nSensor Dropout / Missing Value Rate:")
null_rates = ts_df[['heart_rate', 'respiratory_rate', 'spo2', 'hrv_rmssd_ms']].isna().mean() * 100
print(null_rates.round(2).astype(str) + '%')